# 📈 Análise Exploratória — Mercado de Ações Brasileiro

**Ativos analisados:** WEGE3, PETR4, VALE3  
**Fonte dos dados:** Yahoo Finance (via `yfinance`)  
**Objetivo:** Explorar dados históricos de preços, calcular métricas financeiras e gerar visualizações para embasar insights sobre o desempenho dos ativos.

---

## 0. Configuração do Ambiente

In [ ]:
import sys
import os

# Ajusta o path para importar o pacote 'src' corretamente a partir do notebook
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.data_collector import download_stock_data, load_stock_data
from src.data_processor import process_all_stocks, get_descriptive_stats
from src.visualizer import (
    plot_price_evolution,
    plot_price_comparison,
    plot_cumulative_return,
    plot_return_histograms,
    plot_volatility,
    plot_correlation_heatmap,
    plot_boxplot_returns,
)

# Configurações de exibição
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 120

print('Ambiente configurado com sucesso ✔')

---

## 1. Coleta de Dados

Utilizamos a biblioteca `yfinance` para baixar dados históricos diretamente do Yahoo Finance.  
Os dados são salvos em `/data` para evitar chamadas repetidas à API.

In [ ]:
TICKERS     = ['WEGE3.SA', 'PETR4.SA', 'VALE3.SA']
START_DATE  = '2021-01-01'
END_DATE    = '2024-12-31'
DATA_PATH   = '../data/'
IMAGES_PATH = '../images/'

# Baixa os dados (ou carrega do CSV se já existirem)
import glob
existing_files = glob.glob(os.path.join(DATA_PATH, '*.csv'))
if len(existing_files) >= len(TICKERS):
    print('Carregando dados locais...')
    stock_data = load_stock_data(TICKERS, DATA_PATH)
else:
    print('Baixando dados do Yahoo Finance...')
    stock_data = download_stock_data(TICKERS, START_DATE, END_DATE, DATA_PATH)

print(f'\nAtivos carregados: {list(stock_data.keys())}')

In [ ]:
# Inspeciona os dados de WEGE3 como exemplo
print('Shape:', stock_data['WEGE3.SA'].shape)
stock_data['WEGE3.SA'].head()

---

## 2. Tratamento e Processamento dos Dados

Aplicamos o pipeline completo de processamento:
- Limpeza de nulos
- Retorno diário (%)
- Médias móveis de 20 e 50 dias
- Volatilidade anualizada rolling (21d)
- Retorno acumulado

In [ ]:
processed_data = process_all_stocks(stock_data)

# Verifica as colunas após o processamento
print('Colunas após processamento:')
print(processed_data['WEGE3.SA'].columns.tolist())

---

## 3. Análise Exploratória (EDA)

### 3.1 Estatísticas Descritivas

In [ ]:
stats_df = get_descriptive_stats(processed_data)
display(stats_df)

### 3.2 Correlação entre os Retornos

In [ ]:
returns_df = pd.DataFrame({
    t.replace('.SA', ''): processed_data[t]['Daily_Return']
    for t in TICKERS
})

print('Matriz de Correlação dos Retornos Diários:')
display(returns_df.corr().style.background_gradient(cmap='RdYlGn', vmin=-1, vmax=1))

---

## 4. Visualizações

### 4.1 Evolução do Preço com Médias Móveis

In [ ]:
plot_price_evolution(processed_data, IMAGES_PATH)

### 4.2 Comparação de Desempenho (Base 100)

In [ ]:
plot_price_comparison(processed_data, IMAGES_PATH)

### 4.3 Retorno Acumulado

In [ ]:
plot_cumulative_return(processed_data, IMAGES_PATH)

### 4.4 Histogramas de Retornos Diários

In [ ]:
plot_return_histograms(processed_data, IMAGES_PATH)

### 4.5 Volatilidade Anualizada Rolling

In [ ]:
plot_volatility(processed_data, IMAGES_PATH)

### 4.6 Heatmap de Correlação

In [ ]:
plot_correlation_heatmap(processed_data, IMAGES_PATH)

### 4.7 Boxplot dos Retornos Diários

In [ ]:
plot_boxplot_returns(processed_data, IMAGES_PATH)

---

## 5. Conclusões e Insights

> **Edite esta seção com os insights obtidos após executar as células acima.**

Alguns pontos a considerar:

- **Retorno acumulado:** Qual ativo teve melhor desempenho no período?
- **Volatilidade:** Qual ativo apresentou maior risco? Em quais períodos?
- **Correlação:** As ações tendem a se mover juntas? Isso sugere diversificação?
- **Distribuição dos retornos:** Os retornos são simétricos ou apresentam assimetria (skew)?
- **Médias móveis:** Em algum momento houve cruzamento de MA_20 e MA_50 (golden/death cross)?

---

## 6. Próximos Passos

1. 🖥️ **Dashboard interativo** com Streamlit ou Plotly Dash
2. 🤖 **Modelo preditivo** com LSTM ou Prophet para previsão de preços
3. 📡 **API REST** com FastAPI para servir os dados processados em tempo real